# Module 3: Advanced Relational Analytical Engine & Superstore Mining
**Author:** Rakshit Gupta  
**Course Assessment:** Module 3 — Subqueries, Common Table Expressions (CTEs), and Window Functions Mastery  
**Objective:** Apply Subqueries, CTEs, and Window Functions to analyze sales data from the Superstore dataset.

## 1. Setup & Dependencies

In [ ]:
import pandas as pd
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

# Pretty display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully!")

## 2. Load Dataset into SQLite

In [ ]:
conn = sqlite3.connect('superstore_analytics.db')
engine = conn.cursor()

engine.executescript("""
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS superstore_raw;
""")

engine.execute("""
CREATE TABLE superstore_raw (
    Row_ID INT PRIMARY KEY, Order_ID VARCHAR(20), Order_Date DATE, Ship_Date DATE,
    Ship_Mode VARCHAR(50), Customer_ID VARCHAR(20), Customer_Name VARCHAR(100),
    Segment VARCHAR(50), Country VARCHAR(50), City VARCHAR(50), State VARCHAR(50),
    Postal_Code VARCHAR(20), Region VARCHAR(50), Product_ID VARCHAR(20),
    Category VARCHAR(50), Sub_Category VARCHAR(50), Product_Name VARCHAR(200),
    Sales DECIMAL(10,4), Quantity INT, Discount DECIMAL(5,2), Profit DECIMAL(10,4)
);
""")

paths = ['archive (1).zip/Sample - Superstore.csv', 'Sample - Superstore.csv', 'sample_superstore.csv']
inserted = False
for path in paths:
    if os.path.exists(path):
        staging_df = pd.read_csv(path, encoding='windows-1252')
        staging_df.columns = staging_df.columns.str.replace(' ', '_').str.replace('-', '_')
        staging_df.to_sql('superstore_raw', conn, if_exists='append', index=False)
        print(f"[Pipeline Log] Successfully Ingested rows from '{path}'")
        inserted = True
        break

if not inserted:
    mock_records = [
        (1, 'CA-2016-152156', '2016-11-08', '2016-11-11', 'Second Class', 'CG-12520', 'Claire Gute', 'Consumer', 'United States', 'Henderson', 'Kentucky', '42420', 'South', 'FUR-CH-10002024', 'Furniture', 'Chairs', 'Hon Deluxe Chair', 731.94, 3, 0.0, 219.58),
        (2, 'CA-2016-138688', '2016-06-12', '2016-06-16', 'Second Class', 'DV-13045', 'Darrin Van Huff', 'Corporate', 'United States', 'Los Angeles', 'California', '90036', 'West', 'OFF-BI-10003527', 'Office Supplies', 'Labels', 'Self-Adhesive Labels', 957.57, 2, 0.0, 430.22),
        (3, 'CA-2015-115812', '2015-06-09', '2015-06-14', 'Standard Class', 'BH-11710', 'Brosina Hoffman', 'Consumer', 'United States', 'Los Angeles', 'California', '90036', 'West', 'TEC-CO-10004722', 'Technology', 'Phones', 'iPhone 13 Pro Max', 1706.18, 4, 0.2, 680.12)
    ]
    engine.executemany("INSERT INTO superstore_raw VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", mock_records)

engine.executescript("""
CREATE TABLE customers AS SELECT DISTINCT Customer_ID, Customer_Name, Segment FROM superstore_raw;
CREATE TABLE products AS SELECT DISTINCT Product_ID, Product_Name, Category, Sub_Category FROM superstore_raw;
CREATE TABLE orders AS SELECT Row_ID, Order_ID, Customer_ID, Product_ID, Order_Date, Sales, Quantity, Profit FROM superstore_raw;
""")
conn.commit()
print("[Pipeline Log] 3NF Relational Tables generated generated cleanly.")

## 3. Core Analytical Queries (Subqueries, CTEs, Window Functions)

In [ ]:
print("--- TASK 2.1: Orders Greater than Average Sales ---")
display(pd.read_sql_query("SELECT order_id, customer_id, sales FROM orders WHERE sales > (SELECT AVG(sales) FROM orders) LIMIT 5;", conn))

print("\n--- TASK 2.2: Highest Sales Order for Each Customer ---")
display(pd.read_sql_query("SELECT o1.customer_id, o1.order_id, o1.sales FROM orders o1 WHERE o1.sales = (SELECT MAX(o2.sales) FROM orders o2 WHERE o2.customer_id = o1.customer_id) LIMIT 5;", conn))

print("\n--- TASK 2.3: Total Sales For Each Customer (CTE) ---")
display(pd.read_sql_query("WITH CustomerSales AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id) SELECT c.customer_name, ROUND(cte.ts, 2) as total_sales FROM CustomerSales cte JOIN customers c ON cte.customer_id = c.customer_id LIMIT 5;", conn))

print("\n--- TASK 2.4: Customers Total Sales Above Average ---")
display(pd.read_sql_query("WITH CustomerSales AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id) SELECT c.customer_name, ROUND(cte.ts, 2) FROM CustomerSales cte JOIN customers c ON cte.customer_id = c.customer_id WHERE cte.ts > (SELECT AVG(ts) FROM CustomerSales) LIMIT 5;", conn))

print("\n--- TASK 2.5: Rank All Customers Based on Total Sales ---")
display(pd.read_sql_query("WITH CustomerRank AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id) SELECT c.customer_name, DENSE_RANK() OVER (ORDER BY cte.ts DESC) as customer_rank FROM CustomerRank cte JOIN customers c ON cte.customer_id = c.customer_id LIMIT 5;", conn))

print("\n--- TASK 2.6: Assign Row Numbers Partitioned by Customer ---")
display(pd.read_sql_query("SELECT order_id, customer_id, sales, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY sales DESC) as row_num FROM orders LIMIT 5;", conn))

print("\n--- TASK 2.7: Display Top 3 Customers Based on Total Sales ---")
display(pd.read_sql_query("WITH Ranked AS (SELECT customer_id, SUM(sales) as ts, DENSE_RANK() OVER (ORDER BY SUM(sales) DESC) as rk FROM orders GROUP BY customer_id) SELECT c.customer_name, ROUND(r.ts, 2) as total_sales, r.rk FROM Ranked r JOIN customers c ON r.customer_id = c.customer_id WHERE r.rk <= 3;", conn))

## 4. Step 3 & Mini Project: Final Integrated Dashboard

In [ ]:
print("--- STEP 3: Final Combined Query (JOIN + CTE + Window Function) ---")
q3 = """
WITH Summary AS (SELECT customer_id, SUM(sales) AS ts FROM orders GROUP BY customer_id)
SELECT c.customer_name, ROUND(s.ts, 2) AS total_sales, DENSE_RANK() OVER (ORDER BY s.ts DESC) AS customer_rank
FROM Summary s JOIN customers c ON s.customer_id = c.customer_id ORDER BY customer_rank ASC LIMIT 5;
"""
display(pd.read_sql_query(q3, conn))

print("\n--- MINI PROJECT Q1: Top 5 Customers ---")
display(pd.read_sql_query("SELECT c.customer_name, ROUND(SUM(o.sales), 2) as total_sales FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY o.customer_id ORDER BY total_sales DESC LIMIT 5;", session_db))

print("\n--- MINI PROJECT Q2: Bottom 5 Customers ---")
display(pd.read_sql_query("SELECT c.customer_name, ROUND(SUM(o.sales), 2) as total_sales FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY o.customer_id ORDER BY total_sales ASC LIMIT 5;", conn))

print("\n--- MINI PROJECT Q3: Single-Order Customers ---")
display(pd.read_sql_query("SELECT c.customer_name, COUNT(DISTINCT o.order_id) as order_count FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY o.customer_id HAVING order_count = 1 LIMIT 5;", conn))

print("\n--- MINI PROJECT Q4: Customers with Above-Average Sales ---")
display(pd.read_sql_query("SELECT DISTINCT c.customer_name FROM orders o JOIN customers c ON o.customer_id = c.customer_id WHERE o.sales > (SELECT AVG(sales) FROM orders) LIMIT 5;", conn))

print("\n--- MINI PROJECT Q5: Highest Order Value Per Customer ---")
display(pd.read_sql_query("SELECT c.customer_name, ROUND(MAX(o.sales), 2) as max_single_order FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY o.customer_id LIMIT 5;", conn))
conn.close()